# Etapa 4 - Análise de cobertura por categoria e loja

Notebook executável da Etapa 4. A lógica canônica fica em `etapa4_cobertura_categoria_loja.py`; este notebook reexecuta o script e apresenta os principais arquivos gerados.

## Metodologia resumida

- Fonte principal: snapshot de cobertura da Etapa 2 no grão loja × produto.
- Cruzamento com outputs auditáveis da Etapa 3 para receita, participação e variação anual.
- Visão sempre separada para rede completa e rede física sem Loja 93.
- Priorização operacional compara lojas físicas entre si e lista a Loja 93 em escopo separado.
- `DIAS_COBERTURA` infinito é contado separadamente; médias e medianas usam apenas valores finitos.
- `TRANSACOES` nos cruzamentos com a Etapa 3 representa linhas de venda, não cupons reais.

> **Como ler a cobertura:** o estoque projetado é reconstruído como `estoque inicial + compras − vendas` (não é contagem física). Como a base não tem foto de estoque de abertura para a maioria dos pares e ~88% dos SKUs vendem sem compra registrada, ~91% dos pares ficam com estoque ≤ 0 e são marcados como "EM RUPTURA". Logo, a taxa de ruptura é **conservadora por construção** (erra para sinalizar ruptura a mais) e mede **prioridade relativa de reposição**, não disponibilidade física real. "Receita em risco" = receita histórica já realizada pelos pares hoje em ruptura/crítico, usada para ordenar prioridade — não é perda projetada.

In [1]:
import runpy
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'outputs' / 'etapa4'

runpy.run_path(str(ROOT / 'notebooks' / 'etapa4_cobertura_categoria_loja.py'), run_name='__main__')

Carregando cobertura da Etapa 2 e dimensoes...
Agregando cobertura por categoria, loja e categoria x loja...


Cruzando com outputs auditaveis da Etapa 3...
Validando reconciliacoes numericas...


Salvando arquivos auditaveis...

--- Destaques Etapa 4 ---
Pares totais: 28,721
Pares em ruptura/critico: 26,713 (93.0%)
Receita historica em risco: R$ 464.6M
Validacoes OK: 43/43

[OK] Arquivos salvos em outputs/etapa4/
  outputs\etapa4\cobertura_categoria_loja.csv
  outputs\etapa4\cobertura_categorias_n1.csv
  outputs\etapa4\cobertura_categorias_n2.csv
  outputs\etapa4\cobertura_categorias_n3.csv
  outputs\etapa4\cobertura_lojas.csv
  outputs\etapa4\documentacao_tecnica_etapa4.md
  outputs\etapa4\priorizacao_reposicao_categoria_loja.csv
  outputs\etapa4\recomendacoes_melhoria.csv
  outputs\etapa4\resumo_etapa4.md
  outputs\etapa4\validacoes_etapa4.csv


{'__name__': '__main__',
 '__doc__': '\netapa4_cobertura_categoria_loja.py\n==================================\nEtapa 4 - Analise de cobertura por categoria e loja\n\nObjetivos\n---------\n1. Agregar o snapshot de cobertura da Etapa 2 por categoria (NIVEL_1, NIVEL_2\n   e NIVEL_3), loja e categoria x loja.\n2. Medir pares loja x produto por status de estoque, receita historica em\n   ruptura/critico e dias de cobertura finitos, tratando explicitamente\n   cobertura infinita/sem venda.\n3. Cruzar cobertura com os outputs auditaveis da Etapa 3 para priorizar\n   categorias e lojas de alta receita com baixa cobertura.\n4. Separar rede completa, rede fisica sem Loja 93 e Loja 93 quando a mistura\n   de atacado/B2B com varejo fisico distorceria a leitura.\n5. Gerar recomendacoes e documentar limitacoes metodologicas e de dados.\n\nPremissas e decisoes metodologicas\n----------------------------------\n- O grao de origem e o par loja x produto do snapshot `cobertura_estoque`\n  gerado na Eta

## Arquivos gerados

In [2]:
arquivos = pd.DataFrame(
    {
        'arquivo': [p.name for p in sorted(OUT.glob('*'))],
        'tamanho_kb': [round(p.stat().st_size / 1024, 1) for p in sorted(OUT.glob('*'))],
    }
)
display(arquivos)

,arquivo,tamanho_kb
0,cobertura_categoria_loja.csv,144.3
1,cobertura_categorias_n1.csv,16.5
2,cobertura_categorias_n2.csv,84.2
3,cobertura_categorias_n3.csv,299.2
4,cobertura_lojas.csv,8.6
5,documentacao_tecnica_etapa4.md,4.6
6,priorizacao_reposicao_categoria_loja.csv,103.4
7,recomendacoes_melhoria.csv,2.5
8,resumo_etapa4.md,4.5
9,validacoes_etapa4.csv,4.0


## Validacoes

In [3]:
validacoes = pd.read_csv(OUT / 'validacoes_etapa4.csv')
display(validacoes)
assert (validacoes['STATUS'] == 'OK').all()

,VALIDACAO,OBSERVADO,ESPERADO,DIFERENCA,STATUS
0,Linhas cobertura parquet vs CSV Etapa 2,2.872100e+04,2.872100e+04,0.000000,OK
1,Receita cobertura parquet vs CSV Etapa 2,4.825157e+08,4.825157e+08,0.000000,OK
2,DIAS_COBERTURA negativo inexistente,1.000000e+00,1.000000e+00,0.000000,OK
3,Receita cobertura vs vendas - rede completa,4.825157e+08,4.825157e+08,0.000000,OK
4,Receita cobertura vs vendas - rede fisica,3.292348e+08,3.292348e+08,0.000000,OK
5,categorias_n1 soma RECEITA_HISTORICA_TOTAL - R...,4.825157e+08,4.825157e+08,0.000000,OK
6,categorias_n1 soma PARES_LOJA_PRODUTO - REDE_C...,2.872100e+04,2.872100e+04,0.000000,OK
7,categorias_n1 status soma pares por linha - RE...,1.000000e+00,1.000000e+00,0.000000,OK
8,categorias_n2 soma RECEITA_HISTORICA_TOTAL - R...,4.825157e+08,4.825157e+08,-0.000000,OK
9,categorias_n2 soma PARES_LOJA_PRODUTO - REDE_C...,2.872100e+04,2.872100e+04,0.000000,OK


## Cobertura por categoria

In [4]:
categorias_n1 = pd.read_csv(OUT / 'cobertura_categorias_n1.csv')
cols_cat = [
    'UNIVERSO', 'NIVEL_1', 'RECEITA_HISTORICA_TOTAL', 'RECEITA_RUPTURA_CRITICO',
    'PARES_LOJA_PRODUTO', 'PARES_RUPTURA_CRITICO', 'PCT_PARES_RUPTURA_CRITICO',
    'PART_RECEITA_RISCO_PCT', 'DIAS_COBERTURA_MEDIANA_FINITA', 'RANK_REPOSICAO'
]
display(categorias_n1.query("UNIVERSO == 'REDE_COMPLETA'").head(10)[cols_cat])
display(categorias_n1.query("UNIVERSO == 'REDE_FISICA_SEM_LOJA93'").head(10)[cols_cat])

,UNIVERSO,NIVEL_1,RECEITA_HISTORICA_TOTAL,RECEITA_RUPTURA_CRITICO,PARES_LOJA_PRODUTO,PARES_RUPTURA_CRITICO,PCT_PARES_RUPTURA_CRITICO,PART_RECEITA_RISCO_PCT,DIAS_COBERTURA_MEDIANA_FINITA,RANK_REPOSICAO
0,REDE_COMPLETA,D - ELETROS,1.975705e+08,1.906930e+08,3051,2869,94.034743,96.518943,0.0,1
1,REDE_COMPLETA,C - PISOS E REVESTIMENTOS,5.464366e+07,5.354853e+07,3331,3214,96.487541,97.995861,0.0,2
2,REDE_COMPLETA,R - ELETRONICOS,4.786878e+07,4.754121e+07,281,272,96.797153,99.315694,0.0,3
3,REDE_COMPLETA,I - ILUMINACAO,2.532342e+07,2.280949e+07,2263,1991,87.980557,90.072707,0.0,4
4,REDE_COMPLETA,"K - CAMA, MESA E BANHO",2.321228e+07,2.174579e+07,3623,3483,96.135799,93.682244,0.0,5
5,REDE_COMPLETA,O - MOVEIS,2.272289e+07,2.122458e+07,2035,1853,91.056511,93.406137,0.0,6
6,REDE_COMPLETA,B - UTILIDADES DOMESTICAS,2.251381e+07,2.045144e+07,5545,5032,90.748422,90.839534,0.0,7
7,REDE_COMPLETA,U - METAIS E ACESSORIOS,2.036140e+07,1.963310e+07,1555,1475,94.855305,96.423165,0.0,8
8,REDE_COMPLETA,S - TINTAS E QUIMICOS,1.889943e+07,1.881830e+07,711,699,98.312236,99.570741,0.0,9
9,REDE_COMPLETA,E - MATERIAIS ELETRICOS,1.133240e+07,1.127068e+07,1213,1071,88.293487,99.455413,0.0,10


,UNIVERSO,NIVEL_1,RECEITA_HISTORICA_TOTAL,RECEITA_RUPTURA_CRITICO,PARES_LOJA_PRODUTO,PARES_RUPTURA_CRITICO,PCT_PARES_RUPTURA_CRITICO,PART_RECEITA_RISCO_PCT,DIAS_COBERTURA_MEDIANA_FINITA,RANK_REPOSICAO
23,REDE_FISICA_SEM_LOJA93,D - ELETROS,8.144220e+07,7.879531e+07,2779,2639,94.962217,96.749985,0.0,1
24,REDE_FISICA_SEM_LOJA93,C - PISOS E REVESTIMENTOS,5.464366e+07,5.354853e+07,3064,2947,96.181462,97.995861,0.0,2
25,REDE_FISICA_SEM_LOJA93,I - ILUMINACAO,2.532122e+07,2.280949e+07,2079,1808,86.964887,90.080542,0.0,3
26,REDE_FISICA_SEM_LOJA93,"K - CAMA, MESA E BANHO",2.321228e+07,2.174579e+07,3399,3259,95.881142,93.682244,0.0,4
27,REDE_FISICA_SEM_LOJA93,R - ELETRONICOS,2.113200e+07,2.107991e+07,254,247,97.244094,99.753487,0.0,5
28,REDE_FISICA_SEM_LOJA93,B - UTILIDADES DOMESTICAS,2.251381e+07,2.045144e+07,5111,4598,89.962825,90.839534,0.0,6
29,REDE_FISICA_SEM_LOJA93,U - METAIS E ACESSORIOS,2.036140e+07,1.963310e+07,1423,1343,94.378074,96.423165,0.0,7
30,REDE_FISICA_SEM_LOJA93,S - TINTAS E QUIMICOS,1.889943e+07,1.881830e+07,648,636,98.148148,99.570741,0.0,8
31,REDE_FISICA_SEM_LOJA93,O - MOVEIS,1.364999e+07,1.252467e+07,1849,1687,91.238507,91.755953,0.0,9
32,REDE_FISICA_SEM_LOJA93,E - MATERIAIS ELETRICOS,1.133240e+07,1.127068e+07,1104,962,87.137681,99.455413,0.0,10


## Cobertura por loja

In [5]:
lojas = pd.read_csv(OUT / 'cobertura_lojas.csv')
cols_loja = [
    'UNIVERSO', 'COD_EMPRESA', 'CD_CIDADE', 'CD_ESTADO', 'TIPO_OPERACAO',
    'RECEITA_HISTORICA_TOTAL', 'RECEITA_RUPTURA_CRITICO', 'PARES_LOJA_PRODUTO',
    'PARES_RUPTURA_CRITICO', 'PCT_PARES_RUPTURA_CRITICO', 'RANK_REPOSICAO'
]
display(lojas.query("UNIVERSO == 'REDE_FISICA_SEM_LOJA93'").head(10)[cols_loja])
display(lojas.query("UNIVERSO == 'REDE_COMPLETA' and COD_EMPRESA == 93")[cols_loja])

,UNIVERSO,COD_EMPRESA,CD_CIDADE,CD_ESTADO,TIPO_OPERACAO,RECEITA_HISTORICA_TOTAL,RECEITA_RUPTURA_CRITICO,PARES_LOJA_PRODUTO,PARES_RUPTURA_CRITICO,PCT_PARES_RUPTURA_CRITICO,RANK_REPOSICAO
11,REDE_FISICA_SEM_LOJA93,3,SALVADOR,BA,REDE_FISICA,6.259999e+07,5.867124e+07,2683,2463,91.800224,1
12,REDE_FISICA_SEM_LOJA93,2,RECIFE,PE,REDE_FISICA,5.337709e+07,5.079778e+07,2695,2423,89.907236,2
13,REDE_FISICA_SEM_LOJA93,4,RECIFE,PE,REDE_FISICA,4.330823e+07,4.238388e+07,2692,2554,94.873700,3
14,REDE_FISICA_SEM_LOJA93,6,JOAO PESSOA,PB,REDE_FISICA,3.481680e+07,3.421821e+07,2672,2541,95.097305,4
15,REDE_FISICA_SEM_LOJA93,92,CABO DE SANTO AGOSTINHO,PE,REDE_FISICA,2.856488e+07,2.593386e+07,2365,1968,83.213531,5
16,REDE_FISICA_SEM_LOJA93,5,ARACAJU,SE,REDE_FISICA,2.352973e+07,2.287634e+07,2637,2478,93.970421,6
17,REDE_FISICA_SEM_LOJA93,7,NATAL,RN,REDE_FISICA,2.225043e+07,2.179558e+07,2642,2481,93.906132,7
18,REDE_FISICA_SEM_LOJA93,9,SALVADOR,BA,REDE_FISICA,2.106606e+07,2.106606e+07,2731,2731,100.000000,8
19,REDE_FISICA_SEM_LOJA93,8,CARUARU,PE,REDE_FISICA,2.134132e+07,2.080686e+07,2654,2460,92.690279,9
20,REDE_FISICA_SEM_LOJA93,1,GARANHUNS,PE,REDE_FISICA,1.838021e+07,1.779777e+07,2624,2382,90.777439,10


,UNIVERSO,COD_EMPRESA,CD_CIDADE,CD_ESTADO,TIPO_OPERACAO,RECEITA_HISTORICA_TOTAL,RECEITA_RUPTURA_CRITICO,PARES_LOJA_PRODUTO,PARES_RUPTURA_CRITICO,PCT_PARES_RUPTURA_CRITICO,RANK_REPOSICAO
0,REDE_COMPLETA,93,ALHANDRA,PB,LOJA_93_ATACADO_B2B,1.532810e+08,1.482944e+08,2326,2232,95.958727,1


## Priorizacao categoria x loja

In [6]:
priorizacao = pd.read_csv(OUT / 'priorizacao_reposicao_categoria_loja.csv')
cols_prio = [
    'ESCOPO_PRIORIZACAO', 'RANK_PRIORIDADE_ESCOPO', 'FAIXA_PRIORIDADE',
    'COD_EMPRESA', 'CD_CIDADE', 'CD_ESTADO', 'NIVEL_1', 'RECEITA_RUPTURA_CRITICO',
    'PARES_RUPTURA_CRITICO', 'PCT_PARES_RUPTURA_CRITICO', 'ACAO_RECOMENDADA'
]
display(priorizacao.query("ESCOPO_PRIORIZACAO == 'REDE_FISICA_SEM_LOJA93'").head(15)[cols_prio])
display(priorizacao.query("ESCOPO_PRIORIZACAO == 'LOJA_93_ATACADO_B2B'").head(10)[cols_prio])

,ESCOPO_PRIORIZACAO,RANK_PRIORIDADE_ESCOPO,FAIXA_PRIORIDADE,COD_EMPRESA,CD_CIDADE,CD_ESTADO,NIVEL_1,RECEITA_RUPTURA_CRITICO,PARES_RUPTURA_CRITICO,PCT_PARES_RUPTURA_CRITICO,ACAO_RECOMENDADA
0,REDE_FISICA_SEM_LOJA93,1,ALTA,92,CABO DE SANTO AGOSTINHO,PE,D - ELETROS,1.704283e+07,211,77.859779,Validar saldo físico/transferências e prioriza...
1,REDE_FISICA_SEM_LOJA93,2,ALTA,3,SALVADOR,BA,D - ELETROS,1.557407e+07,248,88.256228,Validar saldo físico/transferências e prioriza...
2,REDE_FISICA_SEM_LOJA93,3,ALTA,3,SALVADOR,BA,C - PISOS E REVESTIMENTOS,1.262309e+07,298,94.303797,Validar saldo físico/transferências e prioriza...
3,REDE_FISICA_SEM_LOJA93,4,ALTA,2,RECIFE,PE,C - PISOS E REVESTIMENTOS,1.070813e+07,312,98.113208,Validar saldo físico/transferências e prioriza...
4,REDE_FISICA_SEM_LOJA93,5,ALTA,2,RECIFE,PE,D - ELETROS,1.061734e+07,276,98.220641,Validar saldo físico/transferências e prioriza...
5,REDE_FISICA_SEM_LOJA93,6,ALTA,4,RECIFE,PE,D - ELETROS,1.000355e+07,276,98.571429,Validar saldo físico/transferências e prioriza...
6,REDE_FISICA_SEM_LOJA93,7,ALTA,6,JOAO PESSOA,PB,D - ELETROS,6.578556e+06,267,96.043165,Validar saldo físico/transferências e prioriza...
7,REDE_FISICA_SEM_LOJA93,8,ALTA,6,JOAO PESSOA,PB,C - PISOS E REVESTIMENTOS,6.358593e+06,292,96.688742,Validar saldo físico/transferências e prioriza...
8,REDE_FISICA_SEM_LOJA93,9,ALTA,8,CARUARU,PE,C - PISOS E REVESTIMENTOS,5.563529e+06,300,97.719870,Validar saldo físico/transferências e prioriza...
9,REDE_FISICA_SEM_LOJA93,10,ALTA,7,NATAL,RN,D - ELETROS,5.385267e+06,273,98.555957,Validar saldo físico/transferências e prioriza...


,ESCOPO_PRIORIZACAO,RANK_PRIORIDADE_ESCOPO,FAIXA_PRIORIDADE,COD_EMPRESA,CD_CIDADE,CD_ESTADO,NIVEL_1,RECEITA_RUPTURA_CRITICO,PARES_RUPTURA_CRITICO,PCT_PARES_RUPTURA_CRITICO,ACAO_RECOMENDADA
213,LOJA_93_ATACADO_B2B,1,ALTA,93,ALHANDRA,PB,D - ELETROS,1.118977e+08,230,84.558824,Validar saldo físico/transferências e prioriza...
214,LOJA_93_ATACADO_B2B,2,MÉDIA,93,ALHANDRA,PB,R - ELETRONICOS,2.646130e+07,25,92.592593,Validar saldo físico/transferências e prioriza...
215,LOJA_93_ATACADO_B2B,3,MONITORAR,93,ALHANDRA,PB,O - MOVEIS,8.699901e+06,166,89.247312,Validar saldo físico/transferências e prioriza...
216,LOJA_93_ATACADO_B2B,4,MONITORAR,93,ALHANDRA,PB,N - INDUSTRIAL E NEGOCIOS,1.235471e+06,44,61.111111,Validar saldo físico/transferências e prioriza...


## Recomendacoes de melhoria

In [7]:
recomendacoes = pd.read_csv(OUT / 'recomendacoes_melhoria.csv')
display(recomendacoes)

,PRIORIDADE,TEMA,LIMITACAO_OU_PROBLEMA,RISCO_ANALITICO,RECOMENDACAO,IMPACTO_ESPERADO
0,ALTA,Movimentações de estoque,A base não contém foto de estoque de abertura ...,O estoque projetado (compras − vendas) fica ≤ ...,"Incorporar foto de estoque de abertura, transf...",Reduzir falsos positivos de ruptura e melhorar...
1,ALTA,Loja 93 e canal,A Loja 93 foi identificada como atacado/B2B po...,Misturar atacado e rede física distorce a cobe...,Criar dimensão de canal/operação e políticas d...,Comparações mais justas e metas de estoque ade...
2,ALTA,Venda média usada na cobertura,A Etapa 2 usa a média dos meses COM venda para...,Itens intermitentes podem ter a velocidade sup...,Rodar análise de sensibilidade com média sobre...,Separar falta crítica de itens recorrentes de ...
3,MÉDIA,Lead time e políticas de estoque,"Não há lead time de fornecedor, estoque mínimo...","A priorização ordena urgência relativa, mas ai...","Adicionar lead time, lote mínimo/múltiplo, cus...",Permitir transformar a priorização em plano de...
4,MÉDIA,Identificador de transação,"A base de vendas não possui id de cupom, pedid...",`TRANSACOES` nos outputs de desempenho represe...,Incluir id_transacao/pedido/nota e número do i...,"Melhorar análises de cesta, recorrência e prio..."


## Resumo interpretativo

In [8]:
display(Markdown((OUT / 'resumo_etapa4.md').read_text(encoding='utf-8')))

# Etapa 4 - Análise de cobertura por categoria e loja

## Glossário rápido (ler antes dos números)

- **Par loja × produto:** cada combinação de uma loja com um produto. É o grão
  desta análise. A rede completa tem 28.721 pares.
- **Cobertura (dias de estoque):** `estoque projetado ÷ venda média mensal × 30`.
  Estima por quantos dias o estoque atende a venda média.
- **Estoque projetado:** **não** é contagem física. É reconstruído como
  `estoque inicial + compras − vendas` na janela jan/2024–dez/2025 (vem da Etapa 2).
- **Em ruptura / crítico:** par com estoque projetado ≤ 0 (ruptura) ou cobertura
  ≤ 30 dias (crítico). São os pares que entram na fila de reposição.
- **Receita histórica em risco:** receita **já realizada no passado** pelos pares
  hoje classificados em ruptura/crítico. É usada para **ordenar prioridade** de
  reposição — **não** é uma previsão de perda futura.

## Como interpretar a cobertura (leia antes de reagir aos 93%)

A taxa de ruptura é altíssima de propósito, por limitação de dados, não por erro:

- A base **não tem foto de estoque de abertura** para a maioria dos pares (entram
  com estoque inicial = 0) e **~88% dos SKUs vendem sem nenhuma compra registrada**.
  Com isso, o estoque projetado fica ≤ 0 na maior parte dos pares, que são então
  marcados como "EM RUPTURA".
- É uma escolha **conservadora deliberada**: na dúvida, o método assume falta de
  estoque — ou seja, **erra para sinalizar ruptura a mais, nunca a menos**.
- Portanto, "93,0% em ruptura/crítico"
  **não** significa "93,0% das prateleiras
  vazias na vida real". É um **ranking de prioridade relativa** de reposição, não a
  taxa real de ruptura física.
- "Disponibilidade física real" exigiria inventário/contagem física, transferências
  entre lojas e ajustes de saldo que **não existem nesta base**.

## Principais achados

- Rede completa: 28.721 pares loja × produto; 26.713 (93,0%) estão em ruptura/crítico (lembrando: prioridade relativa, ver seção acima).
- A receita histórica associada a ruptura/crítico na rede completa é R$ 464,6M, equivalente a 96,3% da receita histórica dos pares. Leia como "concentração de receita por trás da fila de reposição", não como perda projetada.
- Rede física sem Loja 93: 26.395 pares; 24.481 (92,7%) em ruptura/crítico, com R$ 316,3M de receita histórica associada.
- Categoria N1 com maior receita histórica em ruptura/crítico na rede completa: `D - ELETROS`, com R$ 190,7M.
- Categoria N1 com maior receita histórica em ruptura/crítico na rede física: `D - ELETROS`, com R$ 78,8M.
- Loja física com maior pressão de reposição por receita histórica em ruptura/crítico: loja 3 (SALVADOR-BA), com R$ 58,7M.
- Loja 93 deve ser analisada separadamente: no escopo de rede completa, soma R$ 148,3M de receita histórica em ruptura/crítico.
- Maior prioridade na rede física: `D - ELETROS` na loja 92 (CABO DE SANTO AGOSTINHO-PE), com R$ 17,0M de receita histórica associada.
- Maior prioridade da Loja 93: `D - ELETROS`, com R$ 111,9M de receita histórica associada.

## Limitações e cuidados

- A cobertura é uma métrica **conservadora** (erra para o lado de sinalizar ruptura
  a mais) e mede **prioridade relativa**, não disponibilidade física real. Ver a
  seção "Como interpretar a cobertura".
- A base não possui transferências entre lojas, ajustes de inventário nem saldo
  físico posterior ao corte inicial — daí o excesso de ruptura.
- `VENDA_MEDIA_MES` vem da Etapa 2 e usa a média dos meses **com** venda, o que pode
  superestimar a velocidade e subestimar a cobertura de itens intermitentes.
- A Loja 93 é atacado/B2B e não deve ser comparada diretamente com lojas físicas;
  por isso é segregada nas agregações e na priorização.
- `TRANSACOES` nos cruzamentos com a Etapa 3 representa **linhas de venda**, não
  cupons/pedidos/notas reais (a base não tem id de transação).

## Validações

- 43 validações executadas, todas com status `OK`.
- Totais de pares e receita das agregações fecham com `outputs/etapa2/cobertura_estoque.csv`.
- Receita das agregações por categoria e loja fecha com os outputs correspondentes da Etapa 3.
- Loja 93 não aparece no universo `REDE_FISICA_SEM_LOJA93`.
- Estatísticas de dias de cobertura não carregam valores infinitos em médias/medianas.

## Como executar

```bash
cd notebooks
python etapa4_cobertura_categoria_loja.py
```

Os arquivos auditáveis são gravados em `outputs/etapa4/`.
